# Module 5 - Service Tiers

In this notebook you run the expense check on the Priority service tier, so employees who are waiting for a result get it first.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Service tiers on Amazon Bedrock](#s1)
4. [Run the expense check on Priority](#s2)
5. [Choose a tier for each request](#s3)
6. [Try it](#try-it)
7. [Conclusion and key takeaways](#conclusion)
8. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

In Module 4 you made the assistant safe: the guardrail redacts personal data, blocks out-of-policy requests and flags unsupported answers. The expense check is now grounded in the policy and correct, but it is used in two different ways at AnyCompany:

| When | Who is waiting | What matters |
|:--|:--|:--|
| An employee submits a report in the expense tool and waits for the result | A person, looking at the screen | A fast, consistent response |
| Finance rechecks every report at the end of the month | Nobody | Low cost |

Amazon Bedrock lets you choose a **service tier** for each request. In this module you run the check that an employee waits on with the Priority tier, and see how to pick a tier for each kind of request.

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients, set the model ID and find the knowledge base you created in Module 3.

In [ ]:
import json
import re

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name

bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)  # Find the knowledge base
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)  # Query it
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)  # Call models

# Chosen in Module 2 for expense checks. An in-Region model ID (no "global." or "us." prefix):
# requests run in this Region only.
EXPENSE_MODEL_ID = "openai.gpt-oss-120b-1:0"


def find_knowledge_base(name):
    """Return the ID of the knowledge base with this name, or None."""
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]:
        if kb["name"] == name:
            return kb["knowledgeBaseId"]
    return None


# The knowledge base from Module 3, or the prepared one at AWS events.
KNOWLEDGE_BASE_ID = find_knowledge_base("aim317-travel-policy") or find_knowledge_base("aim317-travel-policy-prebuilt")
if KNOWLEDGE_BASE_ID is None:
    print("No knowledge base found. Run Module 3 - Grounding with a Managed Knowledge Base first.")
else:
    print(f"Region:         {REGION}")
    print(f"Model:          {EXPENSE_MODEL_ID}")
    print(f"Knowledge base: {KNOWLEDGE_BASE_ID}")

The cell below defines the grounded expense check from Module 3, with one change: `check_expense` now takes a `tier` argument and passes it to Converse as `serviceTier`. It also tags each request with `requestMetadata`, as in Module 0, so you can find these calls by tier in the invocation logs in Module 6.

Run the cell below to define the helpers and load the expense reports.

In [ ]:
with open("../data/expense_reports.json") as f:
    EXPENSE_REPORTS = json.load(f)


def retrieve(query, number_of_results=5):
    """Return the most relevant policy passages for a query."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"managedSearchConfiguration": {"numberOfResults": number_of_results}},
    )
    return [
        {"text": result["content"]["text"], "source": result["location"]["s3Location"]["uri"].split("/")[-1]}
        for result in response["retrievalResults"]
    ]


def format_passages(passages):
    """Number the passages so the model can cite them."""
    return "\n\n".join(f"[{i}] (source: {p['source']})\n{p['text']}" for i, p in enumerate(passages, start=1))


def format_report(report):
    """Turn an expense report into the text the model reads."""
    lines = [f"Expense report {report['id']} - {report['employee']} - {report['trip']}"]
    for item in report["items"]:
        lines.append(f"- {item['type']}: {item['description']}: ${item['amount']:.2f}")
    return "\n".join(lines)


def get_text(response):
    """Return the text blocks of a Converse response, joined together."""
    return "".join(block["text"] for block in response["output"]["message"]["content"] if "text" in block)


def get_verdict(text):
    """Find the first APPROVED or FLAGGED in a reply."""
    match = re.search(r"\b(APPROVED|FLAGGED)\b", text)
    return match.group(1) if match else "UNKNOWN"


GROUNDED_EXPENSE_PROMPT = (
    "You check AnyCompany expense reports against the AnyCompany policy passages inside the "
    "<policy> tags. Use only those passages. Reply with exactly two lines: first line APPROVED "
    "or FLAGGED, second line a one-sentence reason that cites the passage number, like [1]."
)


def check_expense(report, tier="default"):
    """Check one expense report on the given service tier, and return the verdict and the tier that served it."""
    passages = []
    for item_type in sorted({item["type"] for item in report["items"]}):
        for passage in retrieve(f"AnyCompany {item_type} policy limits for a trip to {report['trip']}", 2):
            if passage["source"] not in [p["source"] for p in passages]:
                passages.append(passage)
    response = bedrock_runtime.converse(
        modelId=EXPENSE_MODEL_ID,
        system=[{"text": GROUNDED_EXPENSE_PROMPT}],
        messages=[{
            "role": "user",
            "content": [{"text": f"<policy>\n{format_passages(passages)}\n</policy>\n\n{format_report(report)}"}],
        }],
        inferenceConfig={"maxTokens": 2000},
        serviceTier={"type": tier},
        requestMetadata={"workshop": "aim317", "module": "05-service-tiers", "feature": "check_expense", "tier": tier},
    )
    # The tier that actually served the request.
    return get_verdict(get_text(response)), response["serviceTier"]["type"]


print(f"{len(EXPENSE_REPORTS)} reports loaded")

## 3. Service tiers on Amazon Bedrock <a id="s1"></a>

Every on-demand request to Amazon Bedrock runs on a [service tier](https://docs.aws.amazon.com/bedrock/latest/userguide/service-tiers-inference.html). You choose the tier per request with the `serviceTier` parameter, so one application can use different tiers for different calls. No reservation or provisioning is required.

| Tier | `serviceTier` type | Price | Use it for |
|:--|:--|:--|:--|
| **Standard** | `default` | Standard on-demand rate | Everyday tasks such as content generation, text analysis and routine document processing. Requests use Standard when you do not set a tier. |
| **Priority** | `priority` | 75% premium over Standard | Latency-sensitive, customer-facing workloads. Priority requests are processed ahead of Standard and Flex requests, for faster responses even during periods of high demand. |
| **Flex** | `flex` | 50% discount from Standard | Workloads that can tolerate longer processing times, such as model evaluations, summarization and multi-step agentic workflows. |

A fourth tier, **Reserved**, provides guaranteed tokens-per-minute capacity for mission-critical workloads, at a fixed monthly price arranged through your AWS account team. See [Amazon Bedrock service tiers](https://aws.amazon.com/bedrock/service-tiers/) and [Amazon Bedrock pricing](https://aws.amazon.com/bedrock/pricing/) for details.

Each model's [model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html) lists the tiers it supports. gpt-oss-120b supports all three tiers above, which makes it a good fit for the expense check.

## 4. Run the expense check on Priority <a id="s2"></a>

Run the cell below to check every report on Priority. Only the `serviceTier` parameter changes from Module 3. Each response includes a `serviceTier` field with the tier that actually served the request, which the cell prints next to the verdict.

In [ ]:
correct = 0
for report in EXPENSE_REPORTS:
    verdict, served_tier = check_expense(report, tier="priority")
    correct += verdict == report["expected_with_policy"]
    print(f"{report['id']}  {verdict:<9} expected {report['expected_with_policy']:<9} served by {served_tier}")

print(f"\n{correct}/{len(EXPENSE_REPORTS)} match the policy")

The verdicts match Module 3: the tier changes how a request is scheduled, not which model answers it or how.

## 5. Choose a tier for each request <a id="s3"></a>

Choose the tier based on whether someone is waiting for the result:

| Request | Tier | Why |
|:--|:--|:--|
| Expense check while an employee waits | **Priority** | The employee is waiting for the result, so the request is latency-sensitive. Priority requests are processed ahead of Standard and Flex requests, including during periods of high demand. |
| End-of-month recheck of every report | **Flex** | No one is waiting for the result, and the job runs at high volume. Flex accepts longer processing times in exchange for a 50% discount. |
| Other requests | **Standard** | Standard provides consistent performance for everyday tasks, and applies when you do not set a tier. |

In Module 6, the deployed `check_expense` request runs on Priority, because an employee waits for it.

## 6. Try it <a id="try-it"></a>

1. **Run the month-end recheck on Flex.** In the cell below, `tier` is set to `flex`. Run it and check the served tier.
2. **Check which tiers a model supports.** Open the [model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html) for Claude Sonnet 5 and for gpt-oss-120b, and compare their Service Tiers sections.

In [ ]:
tier = "flex"

verdict, served_tier = check_expense(EXPENSE_REPORTS[0], tier=tier)
print(f"{EXPENSE_REPORTS[0]['id']}  {verdict}, served by {served_tier}")

## 7. Conclusion and key takeaways <a id="conclusion"></a>

- You choose a **service tier** per request with the `serviceTier` parameter, so one application can use Priority for some calls and Standard or Flex for others.
- **Priority** is priced at a 75% premium over Standard and is processed ahead of other requests. **Flex** is priced at a 50% discount for workloads that can tolerate longer processing times.
- Each response reports the tier that served the request in its `serviceTier` field.
- Each model's card lists the tiers it supports, so check it when you choose a model for a latency-sensitive task.

## 8. Next steps <a id="next-steps"></a>

Your assistant now answers from the policy, stays safe, and checks expenses quickly when someone is waiting. In [Module 6 - Deploy and Monitor](06_deploy_and_monitor.ipynb), you take it out of the notebook: you deploy both capabilities as one AWS Lambda function, then review the invocation logs, latency by service tier, and cost per feature.